# Markov Chains: A Comprehensive Guide

## Introduction

A **Markov chain** is a mathematical system that undergoes transitions from one state to another on a state space. It is a stochastic (random) process that has a very special property: **the future depends only on the present, not on the past**.

### Key Concept: The Markov Property

The **Markov property** (or memorylessness) means:
> The probability of moving to the next state depends only on the current state, not on how we got there.

**Analogy:** It's like having amnesia - you only remember where you are right now, not the journey that brought you here!

## 1. Basic Components of a Markov Chain

### States
- A finite or countable set of possible conditions
- Example: {Sunny, Rainy, Cloudy} for weather

### Transitions
- Movements from one state to another
- Each transition has a probability

### Transition Probability Matrix
- A matrix where element (i,j) represents the probability of moving from state i to state j
- Each row must sum to 1 (probabilities!)

## 2. Simple Example: Weather Model

Let's model weather with three states: Sunny, Rainy, and Cloudy.

**Transition Rules:**
- If today is **Sunny**: 70% chance tomorrow is Sunny, 20% Cloudy, 10% Rainy
- If today is **Cloudy**: 30% chance tomorrow is Sunny, 40% Cloudy, 30% Rainy
- If today is **Rainy**: 20% chance tomorrow is Sunny, 30% Cloudy, 50% Rainy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
import pandas as pd

# Define states
states = ['Sunny', 'Cloudy', 'Rainy']

# Define transition matrix
# Rows: current state, Columns: next state
transition_matrix = np.array([
    [0.7, 0.2, 0.1],  # From Sunny
    [0.3, 0.4, 0.3],  # From Cloudy
    [0.2, 0.3, 0.5]   # From Rainy
])

# Create a nice DataFrame for visualization
df_transition = pd.DataFrame(transition_matrix, 
                             index=states, 
                             columns=states)
print("Transition Probability Matrix:")
display(df_transition)

In [ ]:
# Visualize the transition matrix
plt.figure(figsize=(8, 6))
sns.heatmap(df_transition, annot=True, cmap='YlOrRd', 
            fmt='.2f', cbar_kws={'label': 'Probability'})
plt.title('Weather Transition Probability Matrix', fontsize=14, fontweight='bold')
plt.xlabel('Next State', fontsize=12)
plt.ylabel('Current State', fontsize=12)
plt.tight_layout()
plt.show()

## 3. Simulating a Markov Chain

Let's simulate 30 days of weather starting from a Sunny day!

In [ ]:
def simulate_markov_chain(transition_matrix, states, initial_state, n_steps):
    """
    Simulate a Markov chain.
    
    Parameters:
    - transition_matrix: numpy array of transition probabilities
    - states: list of state names
    - initial_state: starting state (index)
    - n_steps: number of steps to simulate
    
    Returns:
    - sequence: list of states visited
    """
    current_state = initial_state
    sequence = [states[current_state]]
    
    for _ in range(n_steps - 1):
        # Get transition probabilities for current state
        probabilities = transition_matrix[current_state]
        
        # Choose next state based on probabilities
        next_state = np.random.choice(len(states), p=probabilities)
        
        sequence.append(states[next_state])
        current_state = next_state
    
    return sequence

# Simulate 30 days starting from Sunny (index 0)
np.random.seed(42)  # for reproducibility
weather_sequence = simulate_markov_chain(transition_matrix, states, 0, 30)

print("30-Day Weather Simulation:")
print(" -> ".join(weather_sequence))

In [ ]:
# Visualize the sequence
color_map = {'Sunny': 'gold', 'Cloudy': 'lightgray', 'Rainy': 'skyblue'}
colors = [color_map[state] for state in weather_sequence]

plt.figure(figsize=(15, 4))
plt.bar(range(len(weather_sequence)), [1]*len(weather_sequence), color=colors, edgecolor='black')
plt.xticks(range(len(weather_sequence)), [f'Day {i+1}' for i in range(len(weather_sequence))], rotation=45)
plt.yticks([])
plt.title('30-Day Weather Simulation', fontsize=14, fontweight='bold')
plt.xlabel('Day', fontsize=12)

# Add legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=color_map[state], edgecolor='black', label=state) 
                   for state in states]
plt.legend(handles=legend_elements, loc='upper right')
plt.tight_layout()
plt.show()

## 4. Long-Term Behavior: Steady State

One of the most important questions in Markov chains:
> What happens in the long run? Does the distribution of states stabilize?

For many Markov chains, there exists a **steady-state distribution** (also called stationary distribution or equilibrium distribution):
- A probability distribution π where π = π × P
- Once reached, the distribution stays the same
- Independent of the starting state!

In [ ]:
def find_steady_state(transition_matrix, n_iterations=100):
    """
    Find steady state by matrix multiplication.
    """
    # Start with uniform distribution
    state_dist = np.ones(len(transition_matrix)) / len(transition_matrix)
    
    history = [state_dist.copy()]
    
    for _ in range(n_iterations):
        state_dist = state_dist @ transition_matrix
        history.append(state_dist.copy())
    
    return state_dist, history

steady_state, history = find_steady_state(transition_matrix)

print("Steady State Distribution:")
for state, prob in zip(states, steady_state):
    print(f"{state}: {prob:.4f} ({prob*100:.2f}%)")

In [ ]:
# Visualize convergence to steady state
history_array = np.array(history)

plt.figure(figsize=(12, 6))
for i, state in enumerate(states):
    plt.plot(history_array[:, i], label=state, marker='o', markersize=3)

plt.xlabel('Iteration', fontsize=12)
plt.ylabel('Probability', fontsize=12)
plt.title('Convergence to Steady State', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Real-World Example: Simple Text Generation

Markov chains can be used to generate text! We'll build a simple character-level model.

In [ ]:
def build_text_markov_chain(text, order=1):
    """
    Build a Markov chain from text.
    Order 1: next character depends on current character
    """
    transitions = {}
    
    for i in range(len(text) - order):
        current = text[i:i+order]
        next_char = text[i+order]
        
        if current not in transitions:
            transitions[current] = []
        transitions[current].append(next_char)
    
    return transitions

def generate_text(transitions, length=100, seed=None):
    """
    Generate text using the Markov chain.
    """
    if seed is None:
        current = np.random.choice(list(transitions.keys()))
    else:
        current = seed
    
    result = current
    
    for _ in range(length):
        if current in transitions:
            next_char = np.random.choice(transitions[current])
            result += next_char
            current = current[1:] + next_char
        else:
            break
    
    return result

# Sample text
sample_text = """to be or not to be that is the question whether tis nobler in the mind to suffer 
the slings and arrows of outrageous fortune or to take arms against a sea of troubles"""

# Build Markov chain
text_transitions = build_text_markov_chain(sample_text.lower(), order=3)

print("Sample Generated Text (order-3 Markov chain):")
print("=" * 50)
for i in range(3):
    generated = generate_text(text_transitions, length=100, seed="the")
    print(f"\nGeneration {i+1}:")
    print(generated)

## 6. Example: Random Walk

A classic Markov chain: imagine you're on a line with positions 0, 1, 2, 3, 4.
- From any middle position, you move left or right with equal probability
- Positions 0 and 4 are absorbing states (once you reach them, you stay)

In [ ]:
# Random walk transition matrix
random_walk_matrix = np.array([
    [1.0, 0.0, 0.0, 0.0, 0.0],  # State 0 (absorbing)
    [0.5, 0.0, 0.5, 0.0, 0.0],  # State 1
    [0.0, 0.5, 0.0, 0.5, 0.0],  # State 2
    [0.0, 0.0, 0.5, 0.0, 0.5],  # State 3
    [0.0, 0.0, 0.0, 0.0, 1.0]   # State 4 (absorbing)
])

walk_states = ['0', '1', '2', '3', '4']

# Visualize
plt.figure(figsize=(8, 6))
sns.heatmap(random_walk_matrix, annot=True, cmap='Blues', 
            xticklabels=walk_states, yticklabels=walk_states,
            fmt='.2f', cbar_kws={'label': 'Probability'})
plt.title('Random Walk Transition Matrix', fontsize=14, fontweight='bold')
plt.xlabel('Next State', fontsize=12)
plt.ylabel('Current State', fontsize=12)
plt.tight_layout()
plt.show()

# Simulate a walk
np.random.seed(42)
walk = simulate_markov_chain(random_walk_matrix, walk_states, 2, 50)
print("\nRandom Walk Simulation (starting from position 2):")
print(" -> ".join(walk))

In [ ]:
# Visualize multiple random walks
plt.figure(figsize=(14, 6))

for i in range(10):
    walk = simulate_markov_chain(random_walk_matrix, walk_states, 2, 30)
    walk_numeric = [int(s) for s in walk]
    plt.plot(walk_numeric, alpha=0.6, linewidth=2)

plt.xlabel('Step', fontsize=12)
plt.ylabel('Position', fontsize=12)
plt.title('10 Random Walks (starting from position 2)', fontsize=14, fontweight='bold')
plt.yticks([0, 1, 2, 3, 4])
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Properties of Markov Chains

### Irreducibility
- A chain is **irreducible** if you can reach any state from any other state
- Our weather example is irreducible
- The random walk is NOT irreducible (absorbing states!)

### Periodicity
- A state has period k if you can only return to it in multiples of k steps
- **Aperiodic**: period = 1 (can return at any time)

### Ergodicity
- A chain is **ergodic** if it's both irreducible and aperiodic
- Ergodic chains have unique steady states!

## 8. Practical Applications

Markov chains are used in:

1. **Natural Language Processing**
   - Text generation
   - Speech recognition
   - Auto-complete

2. **Finance**
   - Stock price modeling
   - Credit rating transitions

3. **Biology**
   - Gene sequence analysis
   - Population dynamics

4. **Web**
   - PageRank algorithm (Google's original ranking)
   - User behavior modeling

5. **Games**
   - Board game analysis
   - Reinforcement learning (MDPs are Markov chains with rewards!)

## 9. PageRank: A Famous Application

Google's PageRank is essentially a Markov chain on web pages!

Imagine a simplified web with 4 pages:

In [ ]:
# Simple web graph:
# Page A links to B and C
# Page B links to C
# Page C links to A
# Page D links to C

# Transition matrix (following links uniformly at random)
web_matrix = np.array([
    [0.0, 0.5, 0.5, 0.0],  # From A
    [0.0, 0.0, 1.0, 0.0],  # From B
    [1.0, 0.0, 0.0, 0.0],  # From C
    [0.0, 0.0, 1.0, 0.0]   # From D
])

pages = ['Page A', 'Page B', 'Page C', 'Page D']

# Find PageRank (steady state)
pagerank, _ = find_steady_state(web_matrix, n_iterations=50)

print("PageRank Scores:")
for page, score in sorted(zip(pages, pagerank), key=lambda x: x[1], reverse=True):
    print(f"{page}: {score:.4f}")

# Visualize
plt.figure(figsize=(10, 6))
plt.bar(pages, pagerank, color='steelblue', edgecolor='black')
plt.ylabel('PageRank Score', fontsize=12)
plt.title('PageRank for Simple Web Graph', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 10. Summary

### Key Takeaways:

1. **Markov Property**: Future depends only on present, not past
2. **Transition Matrix**: Encodes all the probabilities
3. **Steady State**: Long-term equilibrium distribution
4. **Applications**: Everywhere in ML, NLP, finance, web, and more!

### Mathematical Notation:
- States: $S = \{s_1, s_2, ..., s_n\}$
- Transition probability: $P(X_{t+1} = j | X_t = i) = p_{ij}$
- Steady state: $\pi = \pi P$

### Next Steps:
- Hidden Markov Models (HMMs): when states are hidden
- Markov Decision Processes (MDPs): add actions and rewards
- Monte Carlo Markov Chains (MCMC): for sampling complex distributions

## Exercise: Try It Yourself!

Create your own Markov chain for a scenario of your choice. Some ideas:
1. Student mood throughout the day (energetic, tired, stressed)
2. Traffic light system (red, yellow, green)
3. Stock market (bull, bear, sideways)
4. Game character states (idle, walking, running, jumping)

Use the code cells below to experiment!

In [ ]:
# Your code here!
